# AI 桌面宠物服务端 - 演示笔记本

本笔记本演示 AI 桌面宠物服务端应用的核心功能。

## 目录
1. [概述](#概述)
2. [API测试](#API测试)
3. [记忆系统](#记忆系统)
4. [RAG系统](#RAG系统)
5. [情感分析](#情感分析)

## 概述

AI 桌面宠物服务端使用以下技术构建：
- **FastAPI**: 现代Python Web框架
- **Ollama**: 本地LLM集成
- **ChromaDB**: 用于记忆的向量数据库
- **Redis**: 可选的缓存层

In [ ]:
import requests
import json

# 配置
SERVER_URL = 'http://localhost:8000'

print(f"服务器地址: {SERVER_URL}")

## API测试

### 健康检查

首先，让我们检查服务器是否正在运行。

In [ ]:
# 检查服务器健康状态
try:
    response = requests.get(f"{SERVER_URL}/api/v1/health")
    print(f"服务器状态: {response.json()['status']}")
    print("服务器正在运行！")
except requests.exceptions.ConnectionError:
    print("服务器未运行。请先启动它。")

### 创建会话

In [ ]:
# 创建新会话
response = requests.post(f"{SERVER_URL}/api/v1/session")
session_id = response.json()['session_id']

print(f"会话ID: {session_id}")

### 与AI对话

In [ ]:
# 向AI发送消息
message = "你好！你能做什么？"

response = requests.post(
    f"{SERVER_URL}/api/v1/chat",
    json={
        "session_id": session_id,
        "message": message,
        "stream": False
    }
)

data = response.json()
print(f"AI响应: {data.get('text', '无响应')}")
print(f"\n情感: {data.get('emotion', '未知')}")
print(f"动作: {data.get('action', 'idle')}")
print(f"\n情感向量: {data.get('emotion_vector', {})}")

## 记忆系统

服务器维护三种类型的记忆：
- **工作记忆**: 最近的对话历史
- **长期记忆**: 重要的信息和经历
- **深度记忆**: 用户画像和核心记忆

In [ ]:
# 发送多条消息以构建记忆
messages = [
    "我叫小明",
    "我喜欢编程",
    "我的名字是什么？",
    "你知道关于我的什么？"
]

for msg in messages:
    response = requests.post(
        f"{SERVER_URL}/api/v1/chat",
        json={
            "session_id": session_id,
            "message": msg,
            "stream": False
        }
    )
    data = response.json()
    print(f"\n用户: {msg}")
    print(f"AI: {data.get('text', '无响应')}")

## RAG系统

服务器使用检索增强生成（RAG）来提供关于WSL2的专业知识。

In [ ]:
# 询问WSL2相关问题
wsl2_question = "如何在WSL2中安装新的Linux发行版？"

response = requests.post(
    f"{SERVER_URL}/api/v1/chat",
    json={
        "session_id": session_id,
        "message": wsl2_question,
        "stream": False
    }
)

data = response.json()
print(f"问题: {wsl2_question}")
print(f"\nAI响应: {data.get('text', '无响应')}")
print(f"\n是否使用RAG: {data.get('rag_used', False)}")

## 情感分析

服务器分析情感并将其映射到Live2D参数。

In [ ]:
# 测试不同的情感
test_messages = [
    "我今天很开心！",
    "我对这个消息感到难过",
    "这让我很生气",
    "我很平静"
]

for msg in test_messages:
    response = requests.post(
        f"{SERVER_URL}/api/v1/chat",
        json={
            "session_id": session_id,
            "message": msg,
            "stream": False
        }
    )
    data = response.json()
    print(f"\n消息: {msg}")
    print(f"检测到的情感: {data.get('emotion', '未知')}")
    print(f"Live2D参数: {data.get('live2d_params', {})}")

## WSL2命令模式

服务器可以根据自然语言请求生成WSL2命令。

In [ ]:
# 测试WSL2命令生成
wsl2_requests = [
    "列出当前目录的文件",
    "检查磁盘使用情况",
    "创建一个名为'test'的新目录"
]

for request in wsl2_requests:
    response = requests.post(
        f"{SERVER_URL}/api/v1/chat",
        json={
            "session_id": session_id,
            "message": request,
            "mode": "wsl2",
            "stream": False
        }
    )
    data = response.json()
    print(f"\n请求: {request}")
    print(f"生成的命令: {data.get('wsl2_command', '无命令')}")
    print(f"解释: {data.get('text', '无解释')}")

## 总结

本笔记本演示了：
- 服务器健康检查和会话管理
- 带情感分析的聊天功能
- 记忆系统能力
- 用于专业知识的RAG系统
- WSL2命令生成

更多信息请参考项目文档。